In [1]:
!pip install pyoxigraph
!pip install tqdm

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
data_path = '../../data/raw/dblp.ttl'

In [3]:
from pyoxigraph import Store, RdfFormat
from tqdm import tqdm

# Persistent database stored on disk
store = Store("../../data/raw/dblp_store")

In [4]:


query = """
PREFIX dblp: <https://dblp.org/rdf/schema#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?venue ?type ?name
WHERE {
    ?type rdfs:subClassOf+ dblp:Stream .

    ?venue rdf:type ?type .

    OPTIONAL {
        ?venue dblp:primaryStreamTitle ?name
    }
}
LIMIT 100
"""

results = store.query(query)

conferences_data = {}
journal_data = {}

for row in tqdm(results):
    data = {
        "name": row['name'].value
    }

    if row['type'].value == "https://dblp.org/rdf/schema#Conference":
        conference_id = row['venue'].value.replace('https://dblp.org/streams/conf/', '')
        assert conference_id not in conferences_data, f"Duplicate conference venue found: {row['venue'].value}"
        conferences_data[conference_id] = data
    
    elif row['type'].value == "https://dblp.org/rdf/schema#Journal":
        journal_id = row['venue'].value.replace('https://dblp.org/streams/journals/', '')
        assert journal_id not in journal_data, f"Duplicate journal venue found: {row['venue'].value}"
        journal_data[journal_id] = data

print("5 Conferences examples:")
for i, (venue, data) in enumerate(conferences_data.items()):
    if i >= 5:
        break
    print(f"{venue}: {data}")

print('\n')

print("5 Journals examples:")
for i, (venue, data) in enumerate(journal_data.items()):
    if i >= 5:
        break
    print(f"{venue}: {data}")

100it [00:00, 1043.90it/s]

5 Conferences examples:


5 Journals examples:


In [ ]:

import json

query = """
PREFIX dblp: <https://dblp.org/rdf/schema#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT DISTINCT ?institution ?name ?country ?state ?city
WHERE {
    ?institution rdf:type dblp:Institution .

    OPTIONAL {
    ?institution dblp:institutionName ?institutionName .
    }
    OPTIONAL {
        ?institution dblp:primaryInstitutionName ?primaryName .
    }
    BIND(COALESCE(?primaryName, ?institutionName) AS ?name)

    OPTIONAL {
        ?institution dblp:hasLocation ?location .

        OPTIONAL {
            ?location dblp:countryName ?country .
        }

        OPTIONAL {
            ?location dblp:stateName ?state .
        }

        OPTIONAL {
            ?location dblp:cityName ?city .
        }
    }
}
"""

corrected_duplicates_institutions_data = {

}

institutions_data = {}
results = store.query(query)

for row in results:

    institution_id = row['institution'].value.replace("https://dblp.org/inst/", "")
    data = {
        "name": row['name'].value
    }

    if row['country']:
        data["country"] = row['country'].value
    if row['state']:
        data["state"] = row['state'].value
    if row['city']:
        data["city"] = row['city'].value

    if not institution_id in institutions_data:
        institutions_data[institution_id] = [data]
    else:
        if len(data) > len(institutions_data[institution_id][0]):
            institutions_data[institution_id][0] = data
        elif len(data) == len(institutions_data[institution_id][0]) and data != institutions_data[institution_id][0]:
            institutions_data[institution_id].append(data)

duplicate_institutions = {k: v for k, v in institutions_data.items() if len(v) > 1}

duplicate_institutions_not_corrected = {k: v for k, v in duplicate_institutions.items() if k not in corrected_duplicates_institutions_data}

print(f"Number of institutions with duplicates: {len(duplicate_institutions)}")
print(f"Number of institutions with duplicates not corrected: {len(duplicate_institutions_not_corrected)}")

# print("All duplicate institutions:")
# print(json.dumps(duplicate_institutions, indent=4))

print("Institutions with duplicates not corrected:")
print(json.dumps(duplicate_institutions_not_corrected, indent=4))


8918it [05:15, 28.23it/s]


KeyboardInterrupt: 